# Complexity-erasing language detection PoC (Llama 3.1 8B, free Colab T4)

Two-step, per-paragraph detection of 2 techniques that erase complexity in public debate (a subset of the SemEval / Da San Martino et al. propaganda taxonomy): `black_and_white` (dichotomous reasoning) and `thought_terminating_cliche`.

For each paragraph:
- **Step 1**: extract the discrete claims/assertions made in it
- **Step 2**: for each of the 2 techniques, decide YES/NO, applying explicit "distinction from X" guardrails (see `src/codebook.py`) so the model doesn't flag surface patterns (a short sentence, a rejection of blame) that don't actually fit the definition

Runs **Llama-3.1-8B-Instruct** quantized (GGUF, Q4_K_M) locally via `llama-cpp-python`.

**Before running:** In Colab, go to `Runtime > Change runtime type` and select a **T4 GPU**. Free-tier T4 has ~15GB VRAM, which comfortably fits an 8B model at 4-bit quantization.

This uses a public GGUF conversion of Llama 3.1 8B Instruct, so **no Hugging Face account or token is required**.

## 1. Install dependencies
This installs a prebuilt CUDA wheel of `llama-cpp-python` so GPU offload works without a slow from-source compile.

In [ ]:
!pip install -q huggingface_hub
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
# If the prebuilt wheel above fails to install (Colab's CUDA version can drift over time),
# fall back to compiling from source instead (slower, ~5-10 min):
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python --no-cache-dir --force-reinstall --upgrade

## 2. Clone this repo (codebook, prompt code, and the labeled/target data live here)
The repo needs to be public (or you need to handle auth yourself) for an anonymous clone to work.

In [ ]:
import os

REPO_URL = "https://github.com/hrauxloh/DAAD_Destructive_polarization"
BRANCH = "claude/concept-language-llama-collab-91hjaq"
REPO_DIR = "/content/DAAD_Destructive_polarization"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
import sys
sys.path.insert(0, REPO_DIR)

## 3. Download the quantized model (GGUF)
Using `bartowski/Meta-Llama-3.1-8B-Instruct-GGUF`, a public community conversion. `Q4_K_M` is a good quality/size tradeoff (~4.9GB) for a T4.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"
MODEL_FILE = "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE)
print(model_path)

## 4. Load the model with GPU offload

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # offload all layers to GPU
    n_ctx=4096,
    n_threads=os.cpu_count(),
    verbose=False,
)

## 5. Wire up the codebook, two-step prompt, and a `generate_fn`
`build_chat_messages`, `parse_paragraph_result`, and the codebook all live in `src/` (see `src/codebook.py`, `src/prompting.py`).

In [ ]:
from src.prompting import build_chat_messages, parse_paragraph_result, ParseError, VALID_KEYS

def generate_fn(messages, max_tokens=768, temperature=0.0):
    resp = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp["choices"][0]["message"]["content"]

def analyze_paragraph(paragraph_text, max_retries=2):
    messages = build_chat_messages(paragraph_text)
    last_err = None
    for _ in range(max_retries):
        raw = generate_fn(messages)
        try:
            return parse_paragraph_result(raw)
        except ParseError as e:
            last_err = e
    raise last_err

def print_paragraph_result(paragraph_text, result):
    flagged = {k: v for k, v in result["techniques"].items() if v["present"]}
    print(f'PARAGRAPH: "{paragraph_text[:200]}"')
    print(f'  claims: {result["claims"]}')
    if not flagged:
        print("  no techniques flagged")
    for key, v in flagged.items():
        print(f'  [{key}] "{v["quote"]}"\n    -> {v["rationale"]}')
    print()

## 6. Try it on a single paragraph

In [ ]:
sample_paragraph = (
    "Crime is rising simply because judges have gotten too soft on offenders. "
    "Either we crack down now or our streets will never be safe again."
)

result = analyze_paragraph(sample_paragraph)
print_paragraph_result(sample_paragraph, result)

## 7. Evaluate against the real labeled data (validation, lower priority for now)
`oversimplification_data.csv` (repo root) has ~190 SemEval-derived spans labeled `Black-and-White_Fallacy` or `Thought-terminating_Cliches`. Each row's `span_text` is used directly as the "paragraph" input; there's no source-article context and no negative examples, so this only measures recall/technique-confusion.

A local 8B model is slow on a free T4 — start with a small `sample_size`.

In [ ]:
from src.eval import load_labeled_examples, evaluate, print_report

examples = load_labeled_examples(sample_size=30, seed=0)
result = evaluate(generate_fn, examples=examples)
print_report(result)

## 8. Run over unseen news text (Australian climate-change corpus)
`australia_498sample_climatechange.csv` (repo root) has ~480 full news articles with a `full_text` column — the actual "unseen text" target. Each article's `full_text` is split into paragraphs (`src/paragraphs.py`) and each paragraph is run through the two-step pipeline separately.

In [ ]:
import csv
from src.paragraphs import split_into_paragraphs

with open("australia_498sample_climatechange.csv", newline="", encoding="utf-8") as f:
    aus_articles = list(csv.DictReader(f))

print(f"loaded {len(aus_articles)} articles")

N_ARTICLES = 3
MAX_PARAGRAPHS_PER_ARTICLE = 8   # cap so one article doesn't take forever on free-tier GPU

for article in aus_articles[:N_ARTICLES]:
    paragraphs = split_into_paragraphs(article["full_text"])[:MAX_PARAGRAPHS_PER_ARTICLE]
    print(f"=== {article['document_id']} : {article['title'][:80]} ({len(paragraphs)} paragraphs) ===")
    for p in paragraphs:
        try:
            result = analyze_paragraph(p)
        except ParseError as e:
            print(f'  PARSE_FAIL on paragraph "{p[:80]}": {e}')
            continue
        flagged = {k: v for k, v in result["techniques"].items() if v["present"]}
        if flagged:
            print_paragraph_result(p, result)
    print()

## 9. Run over your own news text
Paste any news text below; it's split into paragraphs and each is analyzed separately.

In [ ]:
news_text = """PASTE NEWS TEXT HERE"""

for p in split_into_paragraphs(news_text):
    result = analyze_paragraph(p)
    print_paragraph_result(p, result)

## Notes / known limitations of this PoC
- Free Colab GPUs are not guaranteed and sessions can disconnect; re-run from the top if that happens.
- The two-step format (extract claims, then judge YES/NO per technique with explicit guardrails) is meant to curb the false positives seen with a flatter free-text extraction prompt (e.g. double-labeling a single sentence with contradictory techniques, or flagging a statement that explicitly rejects binary blame as black-and-white). Check whether it actually does before trusting the output at scale.
- Currently scoped to 2 techniques (`black_and_white`, `thought_terminating_cliche`) by request; `src/codebook.py` previously also covered `causal_oversimplification` and `reductio_ad_hitlerum` if those need to come back in.
- `oversimplification_data.csv` has span-level labels but no source article full-text, so eval in section 7 treats each span in isolation and has no negative (`none`) examples — validation is intentionally deprioritized here; section 8 (the real news corpus) is the priority.
- `temperature=0.0` is used for reproducibility.
- Paragraph splitting (`src/paragraphs.py`) is a simple blank-line/newline heuristic; it may over- or under-segment depending on how the source article's `full_text` was formatted.